In [125]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from torch.utils.data import TensorDataset
from tqdm import tqdm

# *Housing Dataset*

In [126]:
x, y = fetch_california_housing(return_X_y=True, as_frame=True)

x_train, x_testval, y_train, y_testval = train_test_split(x, y, test_size=0.2, random_state=42)
x_test, x_val, y_test, y_val = train_test_split(x_testval, y_testval, test_size=0.5, random_state=42)

x.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude
0,8.3252,41.0,6.984127,1.023810,322.0,2.555556,37.88,-122.23
1,8.3014,21.0,6.238137,0.971880,2401.0,2.109842,37.86,-122.22
2,7.2574,52.0,8.288136,1.073446,496.0,2.802260,37.85,-122.24
3,5.6431,52.0,5.817352,1.073059,558.0,2.547945,37.85,-122.25
4,3.8462,52.0,6.281853,1.081081,565.0,2.181467,37.85,-122.25


# *Pre-processing*

In [127]:
# min_val and max_val are the boundaries for outliers
mean = torch.tensor(x_train.iloc[:, :6].mean().values, dtype=torch.float32)
std = torch.tensor(x_train.iloc[:, :6].std().values, dtype=torch.float32)
min_val = torch.tensor(x_train.iloc[:, :6].min().values, dtype=torch.float32)
max_val = torch.tensor(x_train.iloc[:, :6].max().values, dtype=torch.float32)

class PreprocessingLayer(nn.Module):
    def __init__(self, mean, std, min_val, max_val):
        super().__init__()
        self.register_buffer('mean', mean)
        self.register_buffer('std', std)
        self.register_buffer('min_val', min_val)
        self.register_buffer('max_val', max_val)

    def forward(self, x):
        # Separating latitude and longitude
        x_process = x[:, :6]
        coords = x[:, 6:]
        # Outliers
        x_clipped = torch.clamp(x_process, self.min_val, self.max_val)
        # Standard Scaling - Manually
        x_scaled = (x_clipped - self.mean) / (self.std)
        # Concatenating back coords
        return torch.cat([x_scaled, coords], dim=1)

class HousingPred(nn.Module):
    def __init__(self, mean, std, min_val, max_val, hidden_size):
        super().__init__()
        self.prepro = PreprocessingLayer(mean, std, min_val, max_val)

        self.Model = nn.Sequential(
            nn.Linear(8, hidden_size),
            nn.ReLU(),
            nn.Linear(hidden_size, hidden_size // 2),
            nn.ReLU(),
            nn.Linear(hidden_size // 2, 1)
        )

    def forward(self, x):
        x_ready = self.prepro(x)
        return self.Model(x_ready)

# *Training and evaluation functions*

In [128]:
def train_model(model, train_loader, val_loader, epochs, learning_rate, device):

    optimizer = optim.Adam(model.parameters(), lr = learning_rate)
    loss_func = nn.MSELoss()

    train_losses = []
    val_losses = []

    model.to(device)

    for epoch in range(epochs):
        model.train()
        train_loss = 0.0

        for x_batch, y_batch in tqdm(train_loader, desc=f"Epoch {epoch + 1}/{epochs}", leave=False):
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            optimizer.zero_grad()
            predictions = model(x_batch)
            loss = loss_func(predictions, y_batch)
            loss.backward()
            optimizer.step()
            train_loss += loss.item()

        train_loss /= len(train_loader)
        train_losses.append(train_loss)

        # Validation
        model.eval()
        val_loss = 0.0
        with torch.no_grad():
            for X_batch, y_batch in val_loader:
                X_batch, y_batch = X_batch.to(device), y_batch.to(device)
                predictions = model(X_batch)
                loss = loss_func(predictions, y_batch)
                val_loss += loss.item()

        val_loss /= len(val_loader)
        val_losses.append(val_loss)

        if (epoch + 1) % 50 == 0:
            print(f"Epoch {epoch + 1}/{epochs} - Train Loss: {train_loss:.4f}, Val Loss: {val_loss:.4f}")

    return train_losses, val_losses

def evaluate_model (model, test_loader, device):
    model.eval()
    loss_func = nn.MSELoss()
    total_loss = 0.0
    predictions = []
    targets = []

    with torch.no_grad():
        for x_batch, y_batch in test_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            pred = model(x_batch)
            loss = loss_func(pred, y_batch)
            total_loss += loss.item()
            predictions.append(pred.cpu())
            targets.append(y_batch.cpu())

    mse = total_loss / len(test_loader)
    rmse = torch.sqrt(torch.tensor(mse))

    predictions = torch.cat(predictions)
    targets = torch.cat(targets)
    mae = torch.mean(torch.abs(predictions - targets))

    return rmse, mae, predictions, targets




# *Training*

Preparation before training the models

In [129]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Conversion to tensors
x_train = torch.tensor(x_train.values, dtype=torch.float32)
y_train = torch.tensor(y_train.values, dtype=torch.float32).unsqueeze(1)
x_val = torch.tensor(x_val.values, dtype=torch.float32)
y_val = torch.tensor(y_val.values, dtype=torch.float32).unsqueeze(1)
x_test = torch.tensor(x_test.values, dtype=torch.float32)
y_test = torch.tensor(y_test.values, dtype=torch.float32).unsqueeze(1)

# TensorDatasets
train_dataset = TensorDataset(x_train, y_train)
val_dataset = TensorDataset(x_val, y_val)
test_dataset = TensorDataset(x_test, y_test)

configs = [
    {"name": "Model 1", "hidden_size": 64, "lr": 0.001, "epochs": 50, "batch_size": 16},
    {"name": "Model 2", "hidden_size": 128, "lr": 0.0005, "epochs": 50, "batch_size": 32},
    {"name": "Model 3", "hidden_size": 256, "lr": 0.0001, "epochs": 50, "batch_size": 8}
]

results = {}

This type of implementation to train multiple models with different configurations was used in the previous activity for MNIST dataset. The idea came in a brainstorm with AI, the previous activity wasn't delivered on time, but I took inspiration from it for this implementation.

In [130]:
for config in configs:
    print ("=" * 50)
    print(f"{config['name']}")

    model = HousingPred(mean, std, min_val, max_val, config["hidden_size"])

    # DataLoaders
    train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=config["batch_size"], shuffle=True)
    val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=config["batch_size"])
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=config["batch_size"])

    train_losses, val_losses = train_model(model, train_loader, val_loader, config["epochs"], config["lr"], device)

    rmse, mae, predictions, targets = evaluate_model(model, test_loader, device)

    results[config["name"]] = {"rmse": rmse.item(), "mae": mae.item(), "Train_losses": train_losses, "Val_losses": val_losses, "predictions": predictions, "targets": targets}

    print("\nTest results: ")
    print(f"RMSE: {rmse.item():.4f}")
    print(f"MAE: {mae.item():.4f}\n")

Model 1


Epoch 50/50 - Train Loss: 0.4013, Val Loss: 0.4806

Test results: 
RMSE: 0.6946
MAE: 0.4709

Model 2


Epoch 50/50 - Train Loss: 0.4103, Val Loss: 0.4317

Test results: 
RMSE: 0.6576
MAE: 0.4504

Model 3


Epoch 50/50 - Train Loss: 0.4274, Val Loss: 0.4427

Test results: 
RMSE: 0.6623
MAE: 0.4562



# *Best Model*

In [131]:
best_model_name = min(results.keys(), key=lambda x: results[x]["rmse"])

print("=" * 50)
print(f"Best Model: {best_model_name}")
print(f"RMSE: {results[best_model_name]["rmse"]:.4f}")
print(f"MAE: {results[best_model_name]["mae"]:.4f}")

Best Model: Model 2
RMSE: 0.6576
MAE: 0.4504
